### 1 Set your API key

Get your API key from the Claude Console and set it as an environment variable:

- export ANTHROPIC_API_KEY='your-api-key-here'

To persist the key across shell sessions, add the line to your shell profile (such as ~/.zshrc or ~/.bashrc).

In [ ]:
import anthropic
import json, time
import pandas as pd

client = anthropic.Anthropic()

# NOTE: Claude's guide to prompting: https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/claude-prompting-best-practices


TEMPLATE = """
You will be given a sentence that mixes multiple languages through code-switching. Your task is as follows:

1. Identify which languages are present and determine the ratio of each
2. Normalise the full sentence into English, staying as close to the original sentence's meaning and tone as possible
3. Explain any cultural meaning, tone, or nuance lost in translation
4. Rate your own confidence in the normalisation (0-100)

Sentence: {text}

Respond ONLY in valid JSON, no preamble:
{{
  "languages_detected": ["language1", "language2"],
  "language_ratios": {{"language1": 0.x, "language2": 0.x}},
  "normalised": "...",
  "lost_in_translation": "...",
  "confidence": xx
}}
"""

def prompt_claude(text, max_tokens:int=1000, temp:float=0.3) -> dict:
    message = client.messages.create(
        temperature=temp,
        model="claude-opus-4-7",
        max_tokens=max_tokens,
        system="You are a multilingual linguistics expert.",
        messages=[
            {
                "role": "user",
                "content": TEMPLATE.format(text=text),
            }
        ],
    )
    answer = message.content[0].text
    try:
        return json.loads(answer)
    except json.JSONDecodeError:
        return {"error": answer}

In [ ]:
# Run full dataset
df = pd.read_csv("data/codeswitching_dataset.csv")
results = []

for idx, row in df.iterrows():
    result:dict = prompt_claude(row["text"])
    result["id"] = row["id"]
    result["pair"] = row["pair"]
    result["prompt"] = row["text"]
    results.append(result)
    time.sleep(0.5)  # prevent hitting rate limit

df_results = pd.DataFrame(results)
df_results.to_csv("results/outputs.csv", index=False)

KeyError: 2